# Day 19: CNN Deep Dive — Feature Visualization & Understanding

> **Goal:** Understand *what* a CNN learns by visualising feature maps, filters, and activation patterns at different layers.

---

## Why Visualize?

Neural networks are often called "black boxes." But we can peek inside! Visualization helps you:
- **Debug** — is the network learning useful features?
- **Understand** — what hierarchy of features emerges?
- **Trust** — does the model focus on the right parts of the image?

### What CNNs learn at different depths
| Layer depth | Features detected |
|---|---|
| Early layers | Edges, colors, simple textures |
| Middle layers | Corners, patterns, parts of objects |
| Deep layers | Object parts, semantic features |

In [ ]:
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
import numpy as np

torch.manual_seed(42)
device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

### Setup: Rebuild and Train a CNN

We need a trained model to visualize. Let's rebuild the `SimpleCNN` from Day 17 and train it briefly on CIFAR-10. (If you still have a saved checkpoint, you could load that instead.)

In [ ]:
# Load a trained CNN (rebuild SimpleCNN from Day 17)
class SimpleCNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.block1 = nn.Sequential(
            nn.Conv2d(3, 32, 3, padding=1), nn.BatchNorm2d(32), nn.ReLU(), nn.MaxPool2d(2))
        self.block2 = nn.Sequential(
            nn.Conv2d(32, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2))
        self.block3 = nn.Sequential(
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2))
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Linear(128*4*4, 256), nn.ReLU(), nn.Dropout(0.5), nn.Linear(256, num_classes))

    def forward(self, x):
        x = self.block1(x)
        x = self.block2(x)
        x = self.block3(x)
        return self.classifier(x)

# Quick train (or load checkpoint if available)
CIFAR_MEAN, CIFAR_STD = (0.4914, 0.4822, 0.4465), (0.2470, 0.2435, 0.2616)
test_transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD)])
train_transform = transforms.Compose([
    transforms.RandomHorizontalFlip(), transforms.RandomCrop(32, padding=4),
    transforms.ToTensor(), transforms.Normalize(CIFAR_MEAN, CIFAR_STD)])

train_data = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=train_transform)
test_data  = torchvision.datasets.CIFAR10(root="./data", train=False, download=True, transform=test_transform)
classes = train_data.classes

train_loader = DataLoader(train_data, batch_size=128, shuffle=True)
test_loader  = DataLoader(test_data,  batch_size=128, shuffle=False)

Quick train for 10 epochs — enough to learn meaningful features for visualization (we don't need top accuracy here, just a model that has learned *something*):

In [ ]:
# Train quickly for visualization purposes
model = SimpleCNN().to(device)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
loss_fn = nn.CrossEntropyLoss()

model.train()
for epoch in range(10):
    correct, total = 0, 0
    for imgs, labels in train_loader:
        imgs, labels = imgs.to(device), labels.to(device)
        out = model(imgs)
        loss = loss_fn(out, labels)
        optimizer.zero_grad(); loss.backward(); optimizer.step()
        correct += (out.argmax(1) == labels).sum().item()
        total += labels.size(0)
    if (epoch+1) % 5 == 0:
        print(f"Epoch {epoch+1}: Train Acc = {correct/total:.2%}")

print("Training complete!")

---
## 1. Visualize First-Layer Filters

In [ ]:
filters = model.block1[0].weight.data.cpu()  # (32, 3, 3, 3)

fig, axes = plt.subplots(4, 8, figsize=(14, 7))
for i, ax in enumerate(axes.flat):
    f = filters[i]
    f = (f - f.min()) / (f.max() - f.min() + 1e-8)
    ax.imshow(f.permute(1, 2, 0))
    ax.axis("off")
    ax.set_title(f"F{i}", fontsize=8)
plt.suptitle("Learned First-Layer Filters", fontsize=14)
plt.tight_layout()
plt.show()

---
## 2. Feature Maps at Each Layer

Let's hook into the network and extract intermediate outputs.

In [ ]:
def unnormalize(img, mean=CIFAR_MEAN, std=CIFAR_STD):
    img = img.clone()
    for c in range(3):
        img[c] = img[c] * std[c] + mean[c]
    return img.clamp(0, 1)


def get_feature_maps(model, image):
    """Extract feature maps from each block."""
    features = {}
    model.eval()
    with torch.no_grad():
        x = image.unsqueeze(0).to(device)
        for name in ["block1", "block2", "block3"]:
            x = getattr(model, name)(x)
            features[name] = x.cpu().squeeze(0)
    return features


# Get a test image
img, label = test_data[7]
features = get_feature_maps(model, img)

# Plot
fig, axes_rows = plt.subplots(4, 8, figsize=(16, 8))

# Original image
axes_rows[0][0].imshow(unnormalize(img).permute(1, 2, 0))
axes_rows[0][0].set_title(f"Input ({classes[label]})", fontsize=10)
for j in range(1, 8): axes_rows[0][j].axis("off")

# Feature maps from each block
for row, (name, fmaps) in enumerate(features.items(), 1):
    for j in range(8):
        axes_rows[row][j].imshow(fmaps[j].numpy(), cmap="viridis")
        if j == 0:
            axes_rows[row][j].set_ylabel(name, fontsize=10)
        axes_rows[row][j].set_title(f"Ch {j}", fontsize=7)

for ax in axes_rows.flat:
    ax.axis("off")

plt.suptitle("Feature Maps at Different Layers", fontsize=14)
plt.tight_layout()
plt.show()

---
## 3. Class Activation Mapping (Simple Version)

We can see which spatial regions the network uses for its prediction by looking at the last conv layer's activations weighted by the classifier weights.

In [ ]:
def simple_cam(model, image, target_class):
    """Simple class activation mapping."""
    model.eval()
    with torch.no_grad():
        x = image.unsqueeze(0).to(device)
        x = model.block1(x)
        x = model.block2(x)
        x = model.block3(x)  # (1, 128, 4, 4)
        last_features = x.cpu().squeeze(0)  # (128, 4, 4)

    # Get the weights connecting features to the target class
    fc_weight = model.classifier[1].weight.data.cpu()  # (256, 128*4*4)
    # This is simplified — in practice use proper Grad-CAM
    # We'll just average the feature maps weighted by their activation
    cam = last_features.mean(dim=0)  # (4, 4)

    # Upsample to image size
    cam = torch.nn.functional.interpolate(
        cam.unsqueeze(0).unsqueeze(0), size=(32, 32), mode='bilinear', align_corners=False
    ).squeeze()

    return cam.numpy()


# Display CAM for several images
fig, axes = plt.subplots(2, 5, figsize=(14, 6))

for i in range(10):
    idx = i * 100
    img, label = test_data[idx]
    cam = simple_cam(model, img, label)

    row, col = divmod(i, 5)
    axes[row, col].imshow(unnormalize(img).permute(1, 2, 0))
    axes[row, col].imshow(cam, alpha=0.5, cmap="jet")
    axes[row, col].set_title(classes[label], fontsize=10)
    axes[row, col].axis("off")

plt.suptitle("Feature Activation Heatmaps", fontsize=14)
plt.tight_layout()
plt.show()

---
## 4. Activation Statistics

In [ ]:
# Collect activation statistics across a batch
imgs, labels = next(iter(test_loader))
imgs = imgs.to(device)

model.eval()
activations = {}

with torch.no_grad():
    x = imgs
    for name in ["block1", "block2", "block3"]:
        x = getattr(model, name)(x)
        activations[name] = x.cpu()

# Plot distributions
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for ax, (name, acts) in zip(axes, activations.items()):
    vals = acts.flatten().numpy()
    ax.hist(vals, bins=100, density=True, alpha=0.7)
    ax.set_title(f"{name}\nmean={vals.mean():.3f}, std={vals.std():.3f}")
    ax.set_xlabel("Activation value")
    ax.axvline(0, color="red", linestyle="--", alpha=0.5)
    ax.grid(True, alpha=0.3)

plt.suptitle("Activation Distributions by Layer", fontsize=14)
plt.tight_layout()
plt.show()

---
## 5. Dead Neuron Analysis

In [ ]:
# Check for "dead" ReLU neurons (always output 0)
for name, acts in activations.items():
    # acts shape: (batch, channels, H, W)
    # A channel is "dead" if it outputs 0 for all spatial positions in all samples
    channel_maxes = acts.amax(dim=(0, 2, 3))  # max over batch and spatial
    dead = (channel_maxes == 0).sum().item()
    total_ch = acts.shape[1]
    print(f"{name}: {dead}/{total_ch} dead channels ({dead/total_ch:.1%})")

---
## 🧪 Exercises

### Exercise 1: Compare feature maps
Pick images from 2 different classes (e.g., "cat" and "truck"). Compare their feature maps at block3. Are they different?

### Exercise 2: Max-activating patches
For a given filter in block1, find the 5 image patches from the test set that activate it the most.

### Exercise 3: t-SNE of features
Extract the 256-dimensional features (after flatten, before the final FC layer) for 1000 test images. Plot them with t-SNE, coloured by class.

### Exercise 4: Grad-CAM
Implement proper Gradient-weighted Class Activation Mapping. Use `register_forward_hook` and `register_backward_hook`.

In [ ]:
# Exercises: Your code here

---
## 📝 Key Takeaways

| Concept | Summary |
|---|---|
| **Feature maps** | Each channel at each layer detects a different pattern |
| **Hierarchy** | Early = edges/colors, Middle = textures, Deep = object parts |
| **Class activation maps** | Show which regions the model uses for decisions |
| **Dead neurons** | ReLU neurons that never fire — indicates problems |
| **Visualization is debugging** | Always look at what your model learns! |

**Tomorrow:** Transfer learning — use a pre-trained CNN to get amazing accuracy with very little training!